# 02 — Database Design

### Import Libraries

In [ ]:
import psycopg2
import config

### Database Creation & Connection

In [ ]:
def connect_postgres():
    """ Connect to the postgres database """
    conn = None
    try:
        # read connection parameters
        params = config.config()
        # connect to the PostgreSQL server
        conn = psycopg2.connect(**params)
        conn.autocommit = True
        return conn
    except (Exception, psycopg2.DatabaseError) as error:
        print(error)
        return None

In [ ]:
conn = connect_postgres()
if conn is None:
    raise ConnectionError("Could not connect to the postgres maintenance database.")

try:
    with conn.cursor() as cur:
        # Never drop the warehouse here: doing so erases all populated tables.
        cur.execute("SELECT 1 FROM pg_database WHERE datname = %s", ("divvy_db",))
        if cur.fetchone() is None:
            cur.execute("CREATE DATABASE divvy_db")
        else:
            print("divvy_db already exists; preserving its data.")
finally:
    conn.close()

In [ ]:
def connect_divvy():
    """ Connect to the divvy_db database """
    conn = None
    try:
        # read connection parameters
        params = config.config_divvy()
        # connect to the PostgreSQL server
        conn = psycopg2.connect(**params)
        conn.autocommit = False
        return conn
    except (Exception, psycopg2.DatabaseError) as error:
        print(error)
        return None

### Create Schema

In [ ]:
create_schema = """
    CREATE SCHEMA IF NOT EXISTS divvy;
"""

### Create Dimension Tables

#### Date Dimension

In [ ]:

dim_date = '''
    CREATE TABLE IF NOT EXISTS divvy.dim_date(
        date_key        integer PRIMARY KEY,        -- 20240131
        full_date       date    NOT NULL,
        year            integer NOT NULL,
        quarter         integer NOT NULL,
        month           integer NOT NULL,
        day             integer NOT NULL,
        day_of_week     integer NOT NULL,           -- 1=Monday .. 7=Sunday
        day_name        varchar(10) NOT NULL,
        month_name      varchar(10) NOT NULL,
        is_weekend      boolean NOT NULL
    )
'''

#### Station Dimension

In [ ]:
dim_station = '''
    CREATE TABLE IF NOT EXISTS divvy.dim_station (
        station_key      serial PRIMARY KEY,
        station_id       varchar(50) UNIQUE NOT NULL,
        station_name     varchar(255),
        latitude         double precision,
        longitude        double precision,
        is_active        boolean DEFAULT TRUE
    )
'''

#### Ride Type Dimension

In [ ]:
dim_ride_type = '''
    CREATE TABLE IF NOT EXISTS divvy.dim_ride_type (
        ride_type_key   serial PRIMARY KEY,
        rideable_type   varchar(50) UNIQUE NOT NULL
    )
'''

#### Member Type Dimension

In [ ]:
dim_member_type = '''
    CREATE TABLE IF NOT EXISTS divvy.dim_member_type (
        member_type_key  serial PRIMARY KEY,
        member_casual    varchar(20) UNIQUE NOT NULL   -- 'member' or 'casual'
    )
'''

### Create Fact Table

#### Trip Facts

In [ ]:
fact_trips = '''
    CREATE TABLE IF NOT EXISTS divvy.fact_trip (
        trip_key            bigserial PRIMARY KEY,
        ride_id             varchar(100) NOT NULL UNIQUE,

    -- Foreign keys
        start_date_key      integer NOT NULL,
        end_date_key        integer NOT NULL,
        start_station_key   integer,
        end_station_key     integer,
        ride_type_key       integer NOT NULL,
        member_type_key     integer NOT NULL,

    -- Raw timestamps
        started_at          timestamptz NOT NULL,
        ended_at            timestamptz NOT NULL,

    -- Engineered fields
        duration_minutes    numeric(10,2) NOT NULL,
        start_hour          integer NOT NULL,          -- 0-23
        day_of_week         integer NOT NULL,          -- 1-7, redundant but convenient
        month_partition     integer NOT NULL,          -- e.g. 202401

    -- Geospatial detail at fact level
        start_lat           double precision,
        start_lng           double precision,
        end_lat             double precision,
        end_lng             double precision,

    -- Quality flags
        is_round_trip       boolean,
        is_anomalous        boolean DEFAULT FALSE
    )
'''

#### Foreign Keys Constraints

In [ ]:
add_foreign_keys = '''
    ALTER TABLE divvy.fact_trip
        ADD CONSTRAINT fk_fact_trip_start_date
            FOREIGN KEY (start_date_key) REFERENCES divvy.dim_date(date_key),
        ADD CONSTRAINT fk_fact_trip_end_date
            FOREIGN KEY (end_date_key) REFERENCES divvy.dim_date(date_key),
        ADD CONSTRAINT fk_fact_trip_start_station
            FOREIGN KEY (start_station_key) REFERENCES divvy.dim_station(station_key),
        ADD CONSTRAINT fk_fact_trip_end_station
            FOREIGN KEY (end_station_key) REFERENCES divvy.dim_station(station_key),
        ADD CONSTRAINT fk_fact_trip_ride_type
            FOREIGN KEY (ride_type_key) REFERENCES divvy.dim_ride_type(ride_type_key),
        ADD CONSTRAINT fk_fact_trip_member_type
            FOREIGN KEY (member_type_key) REFERENCES divvy.dim_member_type(member_type_key);
'''

#### Duration sanity checks

In [ ]:
add_sanity_check = '''
    ALTER TABLE divvy.fact_trip
        ADD CONSTRAINT chk_fact_trip_duration_positive
            CHECK (duration_minutes > 0),
        ADD CONSTRAINT chk_fact_trip_start_before_end
            CHECK (started_at < ended_at);
'''

### Indexes for Analysis

#### Time-based analysis

In [ ]:
idx_start_date = '''
    CREATE INDEX IF NOT EXISTS idx_fact_trip_start_date
        ON divvy.fact_trip (start_date_key);
'''

In [ ]:
idx_end_date = '''
    CREATE INDEX IF NOT EXISTS idx_fact_trip_end_date
        ON divvy.fact_trip (end_date_key);
'''

#### Segment and usage analysis

In [ ]:
idx_member_type = '''
    CREATE INDEX IF NOT EXISTS idx_fact_trip_member_type
        ON divvy.fact_trip (member_type_key);
'''

In [ ]:
idx_ride_type = '''
    CREATE INDEX IF NOT EXISTS idx_fact_trip_ride_type
        ON divvy.fact_trip (ride_type_key);
'''

In [ ]:
idx_start_station = '''
    CREATE INDEX IF NOT EXISTS idx_fact_trip_start_station
        ON divvy.fact_trip (start_station_key);
'''

In [ ]:
idx_end_station = '''
    CREATE INDEX IF NOT EXISTS idx_fact_trip_end_station
        ON divvy.fact_trip (end_station_key);
'''

#### Partition-style filtering by month

In [ ]:
idx_month_partition = '''
    CREATE INDEX IF NOT EXISTS idx_fact_trip_month_partition
        ON divvy.fact_trip (month_partition);
'''

### Analytical Views

#### Daily rides by member type

In [ ]:
vw_daily_rides = '''
    CREATE OR REPLACE VIEW divvy.vw_daily_rides AS
    SELECT
        d.full_date, mt.member_casual,
        COUNT(*) AS ride_count,
        AVG(f.duration_minutes) AS avg_duration_minutes
    FROM divvy.fact_trip f
    JOIN divvy.dim_date d ON f.start_date_key = d.date_key
    JOIN divvy.dim_member_type mt ON f.member_type_key = mt.member_type_key
    GROUP BY d.full_date, mt.member_casual;
'''

#### Station-level net flows

In [ ]:
vw_station_flow = '''
    CREATE OR REPLACE VIEW divvy.vw_station_flow AS
    SELECT
        s.station_id,
        s.station_name,
        d.full_date,
        COUNT(CASE WHEN f.start_station_key = s.station_key THEN 1 END) AS rides_started,
        COUNT(CASE WHEN f.end_station_key = s.station_key THEN 1 END) AS rides_ended,
        COUNT(CASE WHEN f.start_station_key = s.station_key THEN 1 END) -
        COUNT(CASE WHEN f.end_station_key = s.station_key THEN 1 END) AS net_outflow
    FROM divvy.dim_station s
    JOIN divvy.fact_trip f 
        ON s.station_key = f.start_station_key
    JOIN divvy.dim_date d 
        ON f.start_date_key = d.date_key
    GROUP BY s.station_id, s.station_name, d.full_date;
'''

### Execute Cursor

In [ ]:
conn = connect_divvy()
if conn is None:
    raise ConnectionError("Could not connect to divvy_db.")

try:
    cursor = conn.cursor()

    # Create schema and tables
    cursor.execute(create_schema)
    cursor.execute(dim_date)
    cursor.execute(dim_station)
    cursor.execute(dim_ride_type)
    cursor.execute(dim_member_type)
    cursor.execute(fact_trips)

    # Add foreign keys and sanity checks
    cursor.execute(add_foreign_keys)
    cursor.execute(add_sanity_check)

    # Create indexes
    cursor.execute(idx_start_date)
    cursor.execute(idx_end_date)
    cursor.execute(idx_member_type)
    cursor.execute(idx_ride_type)
    cursor.execute(idx_start_station)
    cursor.execute(idx_end_station)
    cursor.execute(idx_month_partition)

    # Create views
    cursor.execute(vw_daily_rides)
    cursor.execute(vw_station_flow)

    # Commit the changes to the database
    conn.commit()

finally:
    cursor.close()
    conn.close()